# Business Use Case: Real-World Credit Scoring

End-to-end walkthrough of a production credit scoring segmentation project:

1. Business requirements analysis
2. Data preparation
3. Model development
4. Results validation
5. Deployment considerations

## Business Context

**Scenario:** A bank wants to segment its loan portfolio into risk groups for:
- **Pricing:** Different interest rates by risk tier
- **Monitoring:** Early warning for deteriorating segments
- **Provisioning:** Reserve capital based on segment risk
- **Marketing:** Targeted retention campaigns

**Requirements:**
- Segments must be interpretable: default rate increases with risk tier.
- Segment sizes must remain operationally useful: 5-40% each.
- Predictive power must be acceptable for a coarse segmentation.
- Temporal stability must be validated out-of-time, not optimized on the same sample used to find the cuts.
- Monitoring must separate population drift, score drift, PD drift, and segment-size breaches.


In [1]:
import sys
from pathlib import Path

# Add src to path for importing pso_segmentation
sys.path.insert(0, str(Path("..") / "src"))

import numpy as np
import pandas as pd

from pso_segmentation import (
    OptimizerConfig,
    select_n_segments,
)
from pso_segmentation.segmentation import compute_metrics
from pso_segmentation.segmentation.computation import get_segment_assignments
from pso_segmentation.segmentation.validation import validate_segmentation

np.random.seed(42)
print("Setup complete!")

Setup complete!


## 1. Generate Realistic Portfolio Data

Create synthetic loan portfolio data:


In [2]:
# Portfolio characteristics
n_loans = 50000
start_date = pd.Timestamp("2020-01-01")
end_date = pd.Timestamp("2023-12-31")

# Generate loan features with a balanced temporal spread across several years.
date_grid = pd.date_range(start_date, end_date, freq="D")
origination_dates = np.random.choice(date_grid, size=n_loans, replace=True)

df = pd.DataFrame(
    {
        "loan_id": range(1, n_loans + 1),
        "origination_date": origination_dates,
        "tenor_months": np.random.choice([12, 24, 36, 48, 60], size=n_loans),
    }
)

# Generate risk score (0-1)
df["risk_score"] = np.random.beta(a=2, b=5, size=n_loans)

# Add a mild calendar effect so the synthetic example has a temporal validation problem.
days_since_start = (df["origination_date"] - df["origination_date"].min()).dt.days.astype(float)
time_index = days_since_start / max(days_since_start.max(), 1.0)
calendar_drift = 0.01 * np.sin(2 * np.pi * days_since_start / 365.25) + 0.008 * time_index
default_probability = np.clip(df["risk_score"] + calendar_drift, 0.001, 0.999)

# Generate defaults (correlated with risk score and mildly time-varying)
df["is_default"] = (np.random.rand(n_loans) < default_probability).astype(int)

# Cohorts used only for validation and monitoring, never to fit the cuts.
df = df.sort_values("origination_date").reset_index(drop=True)
df["origination_month"] = df["origination_date"].dt.to_period("M").astype(str)
df["origination_quarter"] = df["origination_date"].dt.to_period("Q").astype(str)
df["origination_year"] = df["origination_date"].dt.year

train_end = df["origination_date"].quantile(0.60)
validation_end = df["origination_date"].quantile(0.80)
df["sample_split"] = np.select(
    [
        df["origination_date"] <= train_end,
        df["origination_date"] <= validation_end,
    ],
    ["train", "temporal_validation"],
    default="oot_test",
)

print("Portfolio Statistics:")
print(df[["tenor_months", "risk_score", "is_default"]].describe())

print("\nTemporal split:")
print(
    df.groupby("sample_split").agg(
        n=("loan_id", "size"),
        start=("origination_date", "min"),
        end=("origination_date", "max"),
        default_rate=("is_default", "mean"),
    )
)
print("\nFirst few records:")
df.head()

Portfolio Statistics:
       tenor_months    risk_score    is_default
count  50000.000000  50000.000000  50000.000000
mean      35.950320      0.285283      0.290360
std       17.017012      0.159931      0.453933
min       12.000000      0.000993      0.000000
25%       24.000000      0.160226      0.000000
50%       36.000000      0.264502      0.000000
75%       48.000000      0.390024      1.000000
max       60.000000      0.927265      1.000000

Temporal split:
                         n      start        end  default_rate
sample_split                                                  
oot_test              9981 2023-03-16 2023-12-31      0.291754
temporal_validation  10009 2022-06-02 2023-03-15      0.293536
train                30010 2020-01-01 2022-06-01      0.288837

First few records:


,loan_id,origination_date,tenor_months,risk_score,is_default,origination_month,origination_quarter,origination_year,sample_split
0,27933,2020-01-01,12,0.137166,0,2020-01,2020Q1,2020,train
1,11460,2020-01-01,48,0.170610,0,2020-01,2020Q1,2020,train
2,46879,2020-01-01,24,0.335103,0,2020-01,2020Q1,2020,train
3,1010,2020-01-01,12,0.183543,0,2020-01,2020Q1,2020,train
4,3612,2020-01-01,24,0.198286,0,2020-01,2020Q1,2020,train


## 2. Define Business-Focused Selection Logic

We define a custom objective function with `compute_metrics` (R² plus monotonicity and size penalties), then use `select_n_segments` to pick the best segmentation while validating temporal stability out-of-time.


In [3]:
BUSINESS_THRESHOLDS = {
    "min_r2": 0.10,
    "min_segment_size": 0.05,
    "max_segment_size": 0.40,
    "population_psi": 0.10,
    "score_psi": 0.10,
    "pd_drift": 0.05,
}


def cohort_column_for_frequency(date_series, frequency):
    if frequency == "M":
        return date_series.dt.to_period("M").astype(str)
    if frequency == "Q":
        return date_series.dt.to_period("Q").astype(str)
    if frequency == "Y":
        return date_series.dt.year.astype(str)
    raise ValueError("frequency must be one of: 'M', 'Q', 'Y'")


def population_stability_index(expected, actual, epsilon=1e-6):
    """Compute PSI between two population distributions."""
    expected = np.asarray(expected, dtype=float)
    actual = np.asarray(actual, dtype=float)

    if expected.sum() <= 0 or actual.sum() <= 0:
        return np.nan

    expected = expected / expected.sum()
    actual = actual / actual.sum()

    expected = np.where(expected == 0, epsilon, expected)
    actual = np.where(actual == 0, epsilon, actual)

    return float(np.sum((expected - actual) * np.log(expected / actual)))


def _segment_distribution(frame, segment_col, n_segments):
    return frame.groupby(segment_col).size().reindex(range(n_segments), fill_value=0).values


def _pd_by_segment(frame, segment_col, label_col, n_segments):
    return (
        frame.groupby(segment_col)[label_col]
        .mean()
        .reindex(range(n_segments), fill_value=np.nan)
        .values
    )


def _score_distribution(scores, bins):
    return np.histogram(scores, bins=bins)[0]


def evaluate_temporal_stability(
    reference_df,
    monitoring_df,
    cuts,
    score_col="risk_score",
    label_col="is_default",
    date_col="origination_date",
    frequency="Q",
    n_score_bins=10,
    thresholds=BUSINESS_THRESHOLDS,
):
    """Evaluate frozen cuts against monitoring cohorts using a fixed reference sample."""
    n_segments = len(np.unique(np.sort(cuts))) + 1
    ref = reference_df.copy()
    mon = monitoring_df.copy()
    ref["risk_segment"] = get_segment_assignments(ref[score_col].values, cuts).astype(int)
    mon["risk_segment"] = get_segment_assignments(mon[score_col].values, cuts).astype(int)
    mon["cohort"] = cohort_column_for_frequency(mon[date_col], frequency)

    ref_population = _segment_distribution(ref, "risk_segment", n_segments)
    ref_pd = _pd_by_segment(ref, "risk_segment", label_col, n_segments)

    quantiles = np.linspace(0, 1, n_score_bins + 1)
    score_edges = np.unique(np.quantile(ref[score_col], quantiles))
    if len(score_edges) < 3:
        score_edges = np.linspace(ref[score_col].min(), ref[score_col].max(), n_score_bins + 1)
    score_edges[0] = -np.inf
    score_edges[-1] = np.inf
    ref_score_distribution = _score_distribution(ref[score_col], score_edges)

    rows = []
    for cohort, cohort_df in mon.groupby("cohort", sort=True):
        population = _segment_distribution(cohort_df, "risk_segment", n_segments)
        cohort_pd = _pd_by_segment(cohort_df, "risk_segment", label_col, n_segments)
        pd_shift = np.abs(cohort_pd - ref_pd)
        total_population = population.sum()
        proportions = population / total_population if total_population > 0 else population
        score_distribution = _score_distribution(cohort_df[score_col], score_edges)

        row = {
            "window": frequency,
            "cohort": cohort,
            "n": int(len(cohort_df)),
            "population_psi": population_stability_index(ref_population, population),
            "score_psi": population_stability_index(ref_score_distribution, score_distribution),
            "pd_drift_mean": float(np.nanmean(pd_shift)),
            "pd_drift_max": float(np.nanmax(pd_shift)),
            "min_segment_share": float(proportions.min()),
            "max_segment_share": float(proportions.max()),
            "monotonic_pd": bool(np.all(np.diff(cohort_pd[~np.isnan(cohort_pd)]) >= 0)),
        }
        row["size_ok"] = bool(
            row["min_segment_share"] >= thresholds["min_segment_size"]
            and row["max_segment_share"] <= thresholds["max_segment_size"]
        )
        row["population_stable"] = bool(row["population_psi"] < thresholds["population_psi"])
        row["score_stable"] = bool(row["score_psi"] < thresholds["score_psi"])
        row["pd_stable"] = bool(row["pd_drift_mean"] < thresholds["pd_drift"])
        row["validation_pass"] = bool(
            row["population_stable"]
            and row["score_stable"]
            and row["pd_stable"]
            and row["size_ok"]
            and row["monotonic_pd"]
        )
        rows.append(row)

    return pd.DataFrame(rows)


def evaluate_temporal_stability_windows(
    reference_df, monitoring_df, cuts, frequencies=("M", "Q", "Y")
):
    frames = [
        evaluate_temporal_stability(reference_df, monitoring_df, cuts, frequency=frequency)
        for frequency in frequencies
    ]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

## 3. Model Development: Package-Driven Selection

The PSO is fit on the development sample only. `select_n_segments` explores a range of segment counts and business parameter grids, then uses a custom selection score that looks at out-of-time stability on a fixed validation cohort.


In [4]:
train_df = df[df["sample_split"] == "train"].copy()
temporal_validation_df = df[df["sample_split"] == "temporal_validation"].copy()
oot_df = df[df["sample_split"] == "oot_test"].copy()


def business_objective_factory(scores, labels, _n_segments, params):
    def objective(cuts):
        try:
            metrics = compute_metrics(scores, labels, cuts)
        except (ValueError, RuntimeError):
            return 0.0

        penalty = 0.0
        if params.get("enforce_monotonic", True) and not metrics.is_monotonic_increasing():
            penalty += params["monotonic_weight"]
        if not metrics.is_balanced(
            min_size=BUSINESS_THRESHOLDS["min_segment_size"],
            max_size=BUSINESS_THRESHOLDS["max_segment_size"],
        ):
            penalty += params["balance_weight"]

        return float(metrics.r2) * (1.0 - penalty)

    return objective


def temporal_selection_score(candidate):
    validation_table = evaluate_temporal_stability_windows(
        train_df,
        temporal_validation_df,
        candidate.cuts,
        frequencies=("M", "Q", "Y"),
    )
    if validation_table.empty:
        return float("-inf")
    if not validation_table["validation_pass"].all():
        return -1.0

    temporal_penalty = np.mean(
        [
            validation_table["population_psi"].mean() / BUSINESS_THRESHOLDS["population_psi"],
            validation_table["score_psi"].mean() / BUSINESS_THRESHOLDS["score_psi"],
            validation_table["pd_drift_mean"].mean() / BUSINESS_THRESHOLDS["pd_drift"],
        ]
    )
    temporal_score = float(np.clip(1.0 - temporal_penalty, 0.0, 1.0))
    train_score = float(np.clip(candidate.metrics.r2 / BUSINESS_THRESHOLDS["min_r2"], 0.0, 1.0))
    return 0.35 * train_score + 0.65 * temporal_score


selection_result = select_n_segments(
    train_df["risk_score"].values,
    train_df["is_default"].values,
    segment_range=(3, 7),
    objective_factory=business_objective_factory,
    base_config=OptimizerConfig(
        pop_size=10,
        max_iter=80,
        w=0.7,
        c1=1.5,
        c2=1.5,
        min_segment_size=BUSINESS_THRESHOLDS["min_segment_size"],
        max_segment_size=BUSINESS_THRESHOLDS["max_segment_size"],
        enforce_monotonic=True,
        track_history=False,
        seed=42,
    ),
    param_grid={
        "monotonic_weight": [0.2, 0.3],
        "balance_weight": [0.15, 0.2],
    },
    selection_func=temporal_selection_score,
    require_valid=True,
)

best_candidate = selection_result.best_candidate
optimizer = best_candidate.optimizer
result = best_candidate.metrics

candidate_summary = pd.DataFrame(
    [
        {
            "n_segments": candidate.n_segments,
            "monotonic_weight": candidate.params["monotonic_weight"],
            "balance_weight": candidate.params["balance_weight"],
            "selection_score": candidate.selection_score,
            "train_r2": candidate.metrics.r2,
            "min_size": candidate.metrics.min_segment_proportion(),
            "max_size": candidate.metrics.max_segment_proportion(),
            "monotonic": candidate.metrics.is_monotonic_increasing(),
            "valid": candidate.valid,
            "validation": candidate.validation_message,
        }
        for candidate in selection_result.candidates
    ]
).sort_values(["selection_score", "train_r2"], ascending=[False, False])

print("Candidate ranking:")
print(candidate_summary.round(4).to_string(index=False))
print(f"\nSelected candidate: {best_candidate.n_segments} segments")
print(f"Train R2: {result.r2:.4f}")
print(f"Train PD by segment: {np.round(result.pd_by_segment, 4)}")
print(f"Selected cuts: {np.round(optimizer.get_cuts(), 4)}")

Candidate ranking:
 n_segments  monotonic_weight  balance_weight  selection_score  train_r2  min_size  max_size  monotonic  valid            validation
          4               0.2            0.15           0.8870    0.1130    0.1254    0.3306       True   True Segmentation is valid
          4               0.2            0.20           0.8870    0.1130    0.1254    0.3306       True   True Segmentation is valid
          4               0.3            0.15           0.8870    0.1130    0.1254    0.3306       True   True Segmentation is valid
          4               0.3            0.20           0.8870    0.1130    0.1254    0.3306       True   True Segmentation is valid
          6               0.2            0.15           0.8848    0.1195    0.0595    0.2757       True   True Segmentation is valid
          6               0.2            0.20           0.8848    0.1195    0.0595    0.2757       True   True Segmentation is valid
          6               0.3            0.15     

## 4. Segment Assignment and Characteristics

The selected cuts are frozen and applied to the full portfolio only for reporting, pricing, export, and monitoring views.


In [5]:
df["risk_segment"] = get_segment_assignments(df["risk_score"].values, optimizer.get_cuts()).astype(
    int
)

segment_profile = (
    df.groupby("risk_segment")
    .agg(
        count=("loan_id", "size"),
        portfolio_share=("loan_id", lambda x: len(x) / len(df)),
        avg_risk_score=("risk_score", "mean"),
        default_rate=("is_default", "mean"),
    )
    .reset_index()
)

print("SEGMENT CHARACTERISTICS - FROZEN CUTS APPLIED TO FULL PORTFOLIO")
print(segment_profile.round(4).to_string(index=False))

# Shared names used by the reporting, pricing, and export sections.
segment_names = [f"Segment {i + 1}" for i in range(result.n_segments)]

SEGMENT CHARACTERISTICS - FROZEN CUTS APPLIED TO FULL PORTFOLIO
 risk_segment  count  portfolio_share  avg_risk_score  default_rate
            0  14770           0.2954          0.1104        0.1171
            1  16445           0.3289          0.2483        0.2524
            2  12409           0.2482          0.3928        0.3966
            3   6376           0.1275          0.5765        0.5828


In [6]:
print("=" * 80)
print("SEGMENT CHARACTERISTICS")
print("=" * 80)
for seg in range(result.n_segments):
    seg_data = df[df["risk_segment"] == seg]
    print(f"{segment_names[seg].upper()}:")
    print("-" * 50)
    print(f"  Count: {len(seg_data)} loans ({len(seg_data) / len(df) * 100:.1f}%)")
    print(f"  Default Rate: {seg_data['is_default'].mean():.1%}")
    print(f"  Avg Risk Score: {seg_data['risk_score'].mean():.3f}")
    print(
        f"  Score Range: [{seg_data['risk_score'].min():.3f}, {seg_data['risk_score'].max():.3f}]"
    )
    print("")

SEGMENT CHARACTERISTICS
SEGMENT 1:
--------------------------------------------------
  Count: 14770 loans (29.5%)
  Default Rate: 11.7%
  Avg Risk Score: 0.110
  Score Range: [0.001, 0.179]

SEGMENT 2:
--------------------------------------------------
  Count: 16445 loans (32.9%)
  Default Rate: 25.2%
  Avg Risk Score: 0.248
  Score Range: [0.179, 0.321]

SEGMENT 3:
--------------------------------------------------
  Count: 12409 loans (24.8%)
  Default Rate: 39.7%
  Avg Risk Score: 0.393
  Score Range: [0.321, 0.482]

SEGMENT 4:
--------------------------------------------------
  Count: 6376 loans (12.8%)
  Default Rate: 58.3%
  Avg Risk Score: 0.577
  Score Range: [0.482, 0.927]



## 6. Business Validation Against Requirements

This validation table separates development quality from out-of-time stability. Temporal checks use the train sample as a fixed reference and evaluate the frozen cuts on the final out-of-time period.


In [7]:
oot_stability = evaluate_temporal_stability_windows(
    train_df,
    oot_df,
    optimizer.get_cuts(),
    frequencies=("M", "Q", "Y"),
)
quarterly_oot = oot_stability[oot_stability["window"] == "Q"].copy()

if quarterly_oot.empty:
    quarterly_oot = oot_stability.copy()

train_validation_ok, train_validation_message = validate_segmentation(
    result,
    min_segment_size=BUSINESS_THRESHOLDS["min_segment_size"],
    max_segment_size=BUSINESS_THRESHOLDS["max_segment_size"],
    monotonic=True,
)

validation_results = [
    {
        "Requirement": "Monotonic default rates on train",
        "Status": "PASS" if result.is_monotonic_increasing() else "FAIL",
        "Details": f"PD by segment: {np.round(result.pd_by_segment, 3)}",
    },
    {
        "Requirement": "Segment sizes on train",
        "Status": "PASS"
        if result.is_balanced(
            BUSINESS_THRESHOLDS["min_segment_size"], BUSINESS_THRESHOLDS["max_segment_size"]
        )
        else "FAIL",
        "Details": f"Range: {result.segment_proportions.min():.1%} to {result.segment_proportions.max():.1%}",
    },
    {
        "Requirement": f"Predictive power on train (R2 >= {BUSINESS_THRESHOLDS['min_r2']:.2f})",
        "Status": "PASS" if result.r2 >= BUSINESS_THRESHOLDS["min_r2"] else "FAIL",
        "Details": f"R2 = {result.r2:.4f}",
    },
    {
        "Requirement": "Package validation on train",
        "Status": "PASS" if train_validation_ok else "FAIL",
        "Details": train_validation_message,
    },
    {
        "Requirement": "Population PSI out-of-time",
        "Status": "PASS" if quarterly_oot["population_stable"].all() else "FAIL",
        "Details": (
            f"max quarterly PSI = {quarterly_oot['population_psi'].max():.4f}; "
            f"threshold = {BUSINESS_THRESHOLDS['population_psi']:.2f}"
        ),
    },
    {
        "Requirement": "Score distribution PSI out-of-time",
        "Status": "PASS" if quarterly_oot["score_stable"].all() else "FAIL",
        "Details": (
            f"max quarterly score PSI = {quarterly_oot['score_psi'].max():.4f}; "
            f"threshold = {BUSINESS_THRESHOLDS['score_psi']:.2f}"
        ),
    },
    {
        "Requirement": "PD drift out-of-time",
        "Status": "PASS" if quarterly_oot["pd_stable"].all() else "FAIL",
        "Details": (
            f"max quarterly mean PD drift = {quarterly_oot['pd_drift_mean'].max():.4f}; "
            f"threshold = {BUSINESS_THRESHOLDS['pd_drift']:.2f}"
        ),
    },
    {
        "Requirement": "PD ordering by out-of-time cohort",
        "Status": "PASS" if quarterly_oot["monotonic_pd"].all() else "FAIL",
        "Details": f"{quarterly_oot['monotonic_pd'].mean():.0%} of quarterly cohorts are monotonic",
    },
    {
        "Requirement": "Segment-size limits by out-of-time cohort",
        "Status": "PASS" if quarterly_oot["size_ok"].all() else "FAIL",
        "Details": (
            f"min share = {quarterly_oot['min_segment_share'].min():.1%}; "
            f"max share = {quarterly_oot['max_segment_share'].max():.1%}"
        ),
    },
    {
        "Requirement": "Individual segment churn",
        "Status": "N/A",
        "Details": "No repeated customer/loan observations across dates in this synthetic origination dataset.",
    },
]

df_val = pd.DataFrame(validation_results)
print("BUSINESS REQUIREMENT VALIDATION:")
print(df_val.to_string(index=False))

all_pass = df_val[df_val["Status"] != "N/A"]["Status"].eq("PASS").all()
print(f"Overall Status: {'ALL REQUIREMENTS MET' if all_pass else 'SOME REQUIREMENTS NOT MET'}")
print("\nQuarterly out-of-time stability details:")
print(
    quarterly_oot[
        [
            "cohort",
            "n",
            "population_psi",
            "score_psi",
            "pd_drift_mean",
            "pd_drift_max",
            "min_segment_share",
            "max_segment_share",
            "monotonic_pd",
            "validation_pass",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

BUSINESS REQUIREMENT VALIDATION:
                              Requirement Status                                                                                    Details
         Monotonic default rates on train   PASS                                                   PD by segment: [0.115 0.251 0.396 0.585]
                   Segment sizes on train   PASS                                                                      Range: 12.5% to 33.1%
   Predictive power on train (R2 >= 0.10)   PASS                                                                                R2 = 0.1130
              Package validation on train   PASS                                                                      Segmentation is valid
               Population PSI out-of-time   PASS                                               max quarterly PSI = 0.0080; threshold = 0.10
       Score distribution PSI out-of-time   PASS                                         max quarterly score PSI = 0.0141; thre

## 7. Pricing Strategy

In [8]:
# Segment-level pricing uplift policy
pricing_uplifts = {0: 0.00, 1: 0.01, 2: 0.03, 3: 0.05, 4: 0.08}
pricing = pd.DataFrame(
    {
        "Segment": [segment_names[i] for i in range(result.n_segments)],
        "Default Rate": np.round(result.pd_by_segment, 3),
        "Count": [len(df[df["risk_segment"] == i]) for i in range(result.n_segments)],
        "Pricing Uplift": [pricing_uplifts[i] for i in range(result.n_segments)],
        "Pricing Action": [
            "No uplift" if pricing_uplifts[i] == 0 else f"Apply +{pricing_uplifts[i]:.0%} uplift"
            for i in range(result.n_segments)
        ],
    }
)
print("RECOMMENDED PRICING STRATEGY:")
print(pricing.to_string(index=False))

print("\nPricing note:")
print(
    "  The segment grid is derived from default risk and can be used to define pricing uplifts without using pre-existing loan-level rates."
)

RECOMMENDED PRICING STRATEGY:
  Segment  Default Rate  Count  Pricing Uplift   Pricing Action
Segment 1         0.115  14770            0.00        No uplift
Segment 2         0.251  16445            0.01 Apply +1% uplift
Segment 3         0.396  12409            0.03 Apply +3% uplift
Segment 4         0.585   6376            0.05 Apply +5% uplift

Pricing note:
  The segment grid is derived from default risk and can be used to define pricing uplifts without using pre-existing loan-level rates.
